# NCCT Stroke Segmentation V9: MAGIC GAN — Contrast-Free CT Perfusion (arXiv:2505.22673v2)

**MAGIC** (Multitask Automated Generation of Intermodal CT perfusion maps) mentranslasikan satu input NCCT ($X$) menjadi **empat peta perfusi sekaligus**:

| Map ($PM$) | Indeks | Makna klinis | Rentang normalisasi (paper) |
|---|---|---|---|
| **CBV** | 0 | Cerebral Blood Volume | [0, 4] mL/100g |
| **CBF** | 1 | Cerebral Blood Flow | [0, 60] mL/100g/min |
| **MTT** | 2 | Mean Transit Time | [0, 12] s |
| **TTP/Tmax** | 3 | Time To Peak / Time-to-Maximum | [0, 25] s |

**Dataset publik:** dataset asli paper bersifat institusional (UF Health, IRB, RAPID — tidak publik). Notebook ini menggunakan **ISLES'24** (`hugging-science/isles24-stroke`, Zenodo DOI 10.5281/zenodo.17652035, lisensi CC BY-NC-SA 4.0) — satu-satunya dataset publik berisi **pasangan NCCT + peta perfusi (CBV, CBF, MTT, Tmax) yang sudah coregistrasi ke ruang NCCT** (analog dengan setup paper). Karena ISLES'24 menyediakan **Tmax** (bukan TTP), Tmax menggantikan TTP sebagai peta ke-4 (keduanya peta *timing*; indeks loss tidak terpengaruh).

**Kesesuaian dengan paper (§2.1 Preprocessing):**
1. **Pseudo-RGB NCCT** — 3 slice NCCT bertetangga (±4 mm, sumbu-z) di-stack sebagai 3 channel.
2. Normalisasi: NCCT dengan **window center 40 HU / width 80 HU**; peta perfusi dengan **rentang fisiologis** di atas.
3. **Shared encoder** → **4 decoder head** (per peta) → modul **PILO**; **PatchGAN 70×70 per peta** (input = peta + channel indikator ones/zeros).

**Objective (persamaan (8) paper):**

$$G^{*}=\arg\min_{G_{CTP}}\max_{D_{CTP}}\mathcal{L}_{GAN}+\lambda_{1}\mathcal{L}_{L1}+\lambda_{2}\mathcal{L}_{EXT}+\lambda_{3}\mathcal{L}_{MML}$$

| # | Loss | Fungsi |
|---|------|--------|
| 1 | **GAN Loss** | Modified Pix2Pix adversarial — BCE rata-rata atas 4 peta & 4 PatchGAN |
| 2 | **L1 Loss** | MAE → structural fidelity |
| 3 | **Extrema Loss** | Bobot error pada ROI ekstrem (jaringan sangat iskemik) |
| 4 | **Multimodal Loss** | Central Volume Principle: $CBV = CBF \times MTT$ |

**Referensi:**
- Khan, W., et al. (2026). "Diagnostically Competitive Performance of a Physiology-Informed Generative Multi-Task Network for Contrast-Free CT Perfusion." *arXiv:2505.22673v2*.
- Riedel, O. E., et al. (2024). "ISLES'24 – A Real-World Longitudinal Multimodal Stroke Dataset." *arXiv:2408.11142*.

### 1. Import Library

In [ ]:
!pip install -q gdown albumentations scikit-image scipy nibabel pyarrow huggingface_hub

In [ ]:
import gdown, zipfile, os, random, time, io, tempfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.metrics import structural_similarity as ssim_fn
from skimage.transform import resize
from scipy import ndimage
import nibabel as nib
import pyarrow.parquet as pq

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | PyTorch {torch.__version__}")

### 2. Persiapan Dataset — ISLES'24 (Hugging Face)

Sesuai paper §2.1.2: **pseudo-RGB NCCT** = tiga slice bertetangga (±4 mm) di-stack sebagai 3 channel, dipasangkan dengan peta perfusi yang sudah **coregistrasi ke ruang NCCT**.

**Sumber:** `hugging-science/isles24-stroke` (mirror publik dataset ISLES'24, lisensi **CC BY-NC-SA 4.0** — non-komersial). Setiap subjek tersimpan sebagai satu file parquet berisi `nii.gz` untuk semua modalitas. Kami mengunduh **subset** (atur `MAX_SUBJECTS`) karena total dataset ~99 GB.

**Pipeline:**
1. **Unduh** `MAX_SUBJECTS` file parquet (satu per subjek).
2. **Preprocess** (mengikuti paper): window NCCT 40/80 HU, normalisasi peta dengan rentang fisiologis, ambil slice aksial tengah, bangun triple pseudo-RGB dengan tetangga ±4 mm, mask latar belakang, resize ke 256×256.
3. **Simpan** sebagai `.npy` float32 [0,1] dalam layout folder:

```
dataset/
├── train/  ├── ncct_prev/  ncct_mid/  ncct_next/
│           ├── cbv/  cbf/  mtt/  tmax/
├── val/    ├── ncct_prev/  ncct_mid/  ncct_next/
│           ├── cbv/  cbf/  mtt/  tmax/
└── test/   ├── ncct_prev/  ncct_mid/  ncct_next/
            ├── cbv/  cbf/  mtt/  tmax/
```

Nama file sama persis antar folder (pasangan per slice).

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "hugging-science/isles24-stroke"
MAX_SUBJECTS = 30            # <-- subset; total tersedia 149 subjek (99 GB penuh)
os.makedirs('dataset_raw', exist_ok=True)
parquet_paths = []
for i in range(MAX_SUBJECTS):
    fn = f"data/train-{i:05d}-of-00149.parquet"
    p = hf_hub_download(REPO, fn, repo_type='dataset', cache_dir='dataset_raw/.cache')
    parquet_paths.append(p)
    print(f"[{i + 1}/{MAX_SUBJECTS}] {fn}")
print(f"\nOK: {len(parquet_paths)} subjek diunduh.")

#### 2.a. Preprocessing — Paper-Faithful (window HU, rentang fisiologis, pseudo-RGB)

In [ ]:
import pyarrow.parquet as pq

PM_ORDER = ['CBV', 'CBF', 'MTT', 'Tmax']          # Tmax menggantikan TTP (ISLES'24)
MAP_RANGES = {'CBV': 4.0, 'CBF': 60.0, 'MTT': 12.0, 'Tmax': 25.0}   # rentang fisiologis (paper)
WINDOW_C, WINDOW_W = 40.0, 80.0                    # window NCCT: center 40, width 80 HU (paper)
N_CENTRAL_SLICES = 10                              # slice aksial tengah per subjek (paper: 10)
OFFSET_MM = 4.0                                    # offset tetangga pseudo-RGB (paper: 4 mm)
TARGET = 256

N_TRAIN = int(MAX_SUBJECTS * 0.7)
N_VAL = int(MAX_SUBJECTS * 0.15)

def load_volume(b):
    with tempfile.NamedTemporaryFile(suffix='.nii.gz', delete=False) as f:
        f.write(b); tmp = f.name
    try:
        return nib.load(tmp).get_fdata().astype(np.float32)
    finally:
        os.unlink(tmp)

def brain_mask(slc, thresh=0.05):
    m = slc > thresh
    m = ndimage.binary_fill_holes(m)
    return ndimage.binary_opening(m, structure=np.ones((5, 5)))

def crop_resize(slc):
    h, w = slc.shape
    s = min(h, w); y0 = (h - s) // 2; x0 = (w - s) // 2
    c = slc[y0:y0 + s, x0:x0 + s]
    return resize(c, (TARGET, TARGET), order=1, preserve_range=True, anti_aliasing=True).astype(np.float32)

def preprocess_subject(row):
    """row: dict dari pq.read_table().to_pydict() (tiap kolom = list 1 elemen). Kembalikan list sample."""
    ncct = np.clip(load_volume(row['ncct'][0]['bytes']), 0.0, WINDOW_W) / WINDOW_W
    maps = {}
    for pm in PM_ORDER:
        d = np.nan_to_num(load_volume(row[pm.lower()][0]['bytes']), nan=0.0, posinf=0.0, neginf=0.0)
        maps[pm] = np.clip(d, 0.0, MAP_RANGES[pm]) / MAP_RANGES[pm]
    nz = ncct.shape[2]
    dz = max(1, int(round(OFFSET_MM / 2.0)))
    zs = np.linspace(dz, nz - 1 - dz, N_CENTRAL_SLICES).round().astype(int)
    samples = []
    for z in zs:
        mid_m = brain_mask(ncct[:, :, z])
        base = f"{row['subject_id'][0]}_{z:03d}"
        entry = {'prev': crop_resize(ncct[:, :, z - dz] * brain_mask(ncct[:, :, z - dz])),
                 'mid':  crop_resize(ncct[:, :, z] * mid_m),
                 'next': crop_resize(ncct[:, :, z + dz] * brain_mask(ncct[:, :, z + dz]))}
        sample = {'base': base, 'ncct': entry}
        for pm in PM_ORDER:
            sample[pm.lower()] = crop_resize((maps[pm][:, :, z] * mid_m).astype(np.float32))
        samples.append(sample)
    return samples

SPLIT_DIRS = {'train': 'dataset/train', 'val': 'dataset/val', 'test': 'dataset/test'}
SUB_DIRS = ['ncct_prev', 'ncct_mid', 'ncct_next'] + [pm.lower() for pm in PM_ORDER]
for sd in SPLIT_DIRS.values():
    for d in SUB_DIRS:
        os.makedirs(os.path.join(sd, d), exist_ok=True)

def split_of(idx):
    if idx < N_TRAIN: return 'train'
    if idx < N_TRAIN + N_VAL: return 'val'
    return 'test'

counts = {'train': 0, 'val': 0, 'test': 0}
for i, p in enumerate(parquet_paths):
    try:
        row = pq.read_table(p).to_pydict()
        samples = preprocess_subject(row)
        sp = split_of(i)
        for s in samples:
            for k, arr in [('ncct_prev', s['ncct']['prev']), ('ncct_mid', s['ncct']['mid']),
                           ('ncct_next', s['ncct']['next'])] + [(pm.lower(), s[pm.lower()]) for pm in PM_ORDER]:
                np.save(os.path.join(SPLIT_DIRS[sp], k, s['base'] + '.npy'), arr)
        counts[sp] += len(samples)
        print(f"[{i + 1}/{len(parquet_paths)}] {samples[0]['base'].split('_')[0]} -> {sp} ({len(samples)} slices)")
    except Exception as e:
        print(f"[{i + 1}/{len(parquet_paths)}] GAGAL: {p} -> {e}")
print(f"\nSelesai. Total sample: {counts}")

### 2.b. Analisis Dataset

Peta sudah dinormalisasi ke [0,1] dengan rentang fisiologis masing-masing. Cek distribusi & jumlah sample per split.

In [ ]:
def analyze_dataset(base_dir):
    for sp in ('train', 'val', 'test'):
        d = os.path.join(base_dir, sp, 'ncct_mid')
        n = len(os.listdir(d)) if os.path.isdir(d) else 0
        print(f"{sp:<6}: {n} slice")
    print()
    print(f"{'Map':<6}|{'mean':>8}|{'std':>8}|{'min':>8}|{'max':>8}")
    print('-' * 44)
    for pm in PM_ORDER:
        pm_dir = os.path.join(base_dir, 'train', pm.lower())
        files = sorted(os.listdir(pm_dir))
        vals = np.concatenate([np.load(os.path.join(pm_dir, f)).ravel() for f in files])
        print(f"{pm:<6}|{vals.mean():8.3f}|{vals.std():8.3f}|{vals.min():8.3f}|{vals.max():8.3f}")

analyze_dataset('dataset')

### 3. Dataset V9 — Paired Pseudo-RGB NCCT + CTP

`NCCTDatasetV9` mengembalikan dict `{'x': NCCT pseudo-RGB (3,H,W), 'maps': 4 peta (4,H,W)}` urutan channel = `PM_ORDER`. Data tersimpan sebagai `.npy` float32 (sudah ternormalisasi [0,1]) — tidak ada pembagian 255 lagi.

Transformasi geometri **sama** diterapkan ke ketiga slice NCCT + keempat peta (via `additional_targets`) agar tetap aligned.

In [ ]:
class NCCTDatasetV9(Dataset):
    """Returns dict: {'x': pseudo-RGB NCCT (3,H,W) = [prev, mid, next], 'maps': 4 peta (4,H,W)}."""
    def __init__(self, split_dir, limit_size=None, transform=None):
        self.split_dir = split_dir
        self.transform = transform
        self.map_dirs = {pm: os.path.join(split_dir, pm.lower()) for pm in PM_ORDER}
        self.ncct_dirs = {k: os.path.join(split_dir, f'ncct_{k}') for k in ('prev', 'mid', 'next')}
        self.three_slice = all(os.path.isdir(d) for d in self.ncct_dirs.values())
        self.ncct_dir = os.path.join(split_dir, 'ncct')
        missing = [d for d in list(self.map_dirs.values()) if not os.path.isdir(d)]
        if not self.three_slice and not os.path.isdir(self.ncct_dir):
            missing += [self.ncct_dir]
        if missing:
            raise FileNotFoundError(f"Folder tidak ditemukan: {missing}. Jalankan sel preprocessing dulu.")
        self.fnames = sorted(os.listdir(self.ncct_dirs['mid'] if self.three_slice else self.ncct_dir))
        if limit_size is not None:
            self.fnames = self.fnames[:limit_size]
    def __len__(self):
        return len(self.fnames)
    def _load(self, d, f):
        p = os.path.join(d, f)
        if p.endswith('.npy'):
            return np.load(p)
        return np.array(Image.open(p).convert('L')).astype(np.float32) / 255.0
    def __getitem__(self, idx):
        f = self.fnames[idx]
        if self.three_slice:
            slices = {k: self._load(self.ncct_dirs[k], f) for k in ('prev', 'mid', 'next')}
        else:
            s = self._load(self.ncct_dir, f)
            slices = {'prev': s, 'mid': s, 'next': s}
        map_np = {pm: self._load(self.map_dirs[pm], f) for pm in PM_ORDER}
        if self.transform is not None:
            kw = {'image': slices['mid'], 'prev': slices['prev'], 'next': slices['next'],
                  **{pm.lower(): map_np[pm] for pm in PM_ORDER}}
            aug = self.transform(**kw)

            def _t(k):
                t = aug[k].float()
                return t[0] if t.ndim == 3 else t   # normalisasi ke 2D (H,W)

            x = torch.stack([_t('prev'), _t('image'), _t('next')], 0)
            maps = torch.stack([_t(pm.lower()) for pm in PM_ORDER], 0)
        else:
            x = torch.from_numpy(np.stack([slices['prev'], slices['mid'], slices['next']], 0))
            maps = torch.from_numpy(np.stack([map_np[pm] for pm in PM_ORDER], 0))
        return {'x': x, 'maps': maps}

def get_dataloaders_v9(base_dir, batch_size=4, limit_size=None):
    targets = {'prev': 'mask', 'next': 'mask', **{pm.lower(): 'mask' for pm in PM_ORDER}}
    trn_tfm = A.Compose([
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        ToTensorV2(),
    ], additional_targets=targets)
    val_tfm = A.Compose([A.Resize(256, 256), ToTensorV2()], additional_targets=targets)
    trn = NCCTDatasetV9(os.path.join(base_dir, 'train'), limit_size, trn_tfm)
    val = NCCTDatasetV9(os.path.join(base_dir, 'val'), limit_size, val_tfm)
    tst = NCCTDatasetV9(os.path.join(base_dir, 'test'), limit_size, val_tfm)
    return (DataLoader(trn, batch_size, shuffle=True),
            DataLoader(val, batch_size, shuffle=False),
            DataLoader(tst, batch_size, shuffle=False))

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders_v9('dataset')
batch = next(iter(val_loader))
print(f"NCCT pseudo-RGB: {batch['x'].shape}   Maps: {batch['maps'].shape}")
print(f"Range NCCT: [{batch['x'].min():.3f}, {batch['x'].max():.3f}]   Maps: [{batch['maps'].min():.3f}, {batch['maps'].max():.3f}]")

def viz_sample9(x, maps, n=2):
    n = min(n, x.shape[0])
    fig, axes = plt.subplots(n, 7, figsize=(24, 4 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        for k, lbl in zip(range(3), ['NCCT prev', 'NCCT mid', 'NCCT next']):
            axes[i, k].imshow(x[i, k].cpu().numpy(), cmap='gray')
            axes[i, k].set_title(lbl)
        for j, pm in enumerate(PM_ORDER):
            axes[i, 3 + j].imshow(maps[i, j].cpu().numpy(), cmap='viridis')
            axes[i, 3 + j].set_title(f'GT {pm}')
        for j in range(7):
            axes[i, j].axis('off')
    plt.tight_layout(); plt.show()

viz_sample9(batch['x'], batch['maps'])

### 4. Model Arsitektur — Sesuai Paper

**Generator (MAGIC, paper-faithful):**
1. **Shared encoder** — serangkaian conv (encoding) yang menangkap fitur hierarkis dari pseudo-RGB NCCT.
2. **Empat decoder head terpisah** (satu per tipe peta) — 3 level up-sampling dengan skip connection dari shared encoder, menghasilkan peta 256×256×1.
3. **PILO (Physician-in-the-Loop)** — peta mentah tiap head dikonkatenasi dengan luminance NCCT lalu di-reduce via konvolusi 1×1; rasio struktur-NCCT vs perfusi-sintetik dapat diatur **interaktif saat inferensi** (α).

**Discriminator:** **PatchGAN 70×70 per tipe peta**, input *conditional*: `cat([peta, indikator])` — channel indikator = **ones** (real) / **zeros** (fake), persis seperti paper.

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

class DecoderHead(nn.Module):
    """Decoder spesifik per tipe peta: 3 level up-sampling + skip dari shared encoder."""
    def __init__(self):
        super().__init__()
        self.up1 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.b1 = conv_block(128 + 256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.b2 = conv_block(64 + 128, 64)
        self.up3 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.b3 = conv_block(32 + 64, 32)
        self.out = nn.Conv2d(32, 1, 1)
    def forward(self, b, s3, s2, s1):
        d = self.b1(torch.cat([self.up1(b), s3], 1))
        d = self.b2(torch.cat([self.up2(d), s2], 1))
        d = self.b3(torch.cat([self.up3(d), s1], 1))
        return self.out(d)

class MAGICGenerator(nn.Module):
    """Paper: shared encoder + 4 task-specific decoder heads + PILO fusion.
    Input pseudo-RGB NCCT (3,256,256) -> output 4 peta (4,256,256) [CBV, CBF, MTT, Tmax]."""
    def __init__(self, in_ch=3, n_maps=4):
        super().__init__()
        self.n_maps = n_maps
        self.e0 = nn.Conv2d(in_ch, 32, 3, padding=1)
        self.e1 = conv_block(32, 64)
        self.e2 = conv_block(64, 128)
        self.e3 = conv_block(128, 256)
        self.pool = nn.MaxPool2d(2, 2)
        self.heads = nn.ModuleList([DecoderHead() for _ in range(n_maps)])
        self.luminance = nn.Conv2d(in_ch, 1, 1)          # struktur NCCT untuk PILO
        self.pilo = nn.ModuleList([nn.Conv2d(2, 1, 1) for _ in range(n_maps)])
    def forward(self, x):
        s0 = F.relu(self.e0(x))
        s1 = self.e1(s0); p1 = self.pool(s1)
        s2 = self.e2(p1); p2 = self.pool(s2)
        s3 = self.e3(p2); b = self.pool(s3)
        lum = self.luminance(x)
        outs = []
        for i in range(self.n_maps):
            raw = self.heads[i](b, s3, s2, s1)
            outs.append(self.pilo[i](torch.cat([raw, lum], 1)))
        return torch.cat(outs, dim=1)

class PatchGANDiscriminator(nn.Module):
    """PatchGAN 70x70 conditional — input: cat([peta, indikator ones/zeros]) = 2 channel."""
    def __init__(self, in_ch=2):
        super().__init__()
        def cblk(i, o, stride, bn=True):
            l = [nn.Conv2d(i, o, 4, stride, 1)]
            if bn:
                l.append(nn.BatchNorm2d(o))
            l.append(nn.LeakyReLU(0.2, inplace=True))
            return l
        self.net = nn.Sequential(
            *cblk(in_ch, 64, 2, bn=False),
            *cblk(64, 128, 2),
            *cblk(128, 256, 2),
            *cblk(256, 512, 1),
            nn.Conv2d(512, 1, 4, 1, 1),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# Sanity check arsitektur
G = MAGICGenerator().to(device)
D = PatchGANDiscriminator().to(device)
x = torch.randn(2, 3, 256, 256).to(device)          # pseudo-RGB NCCT
fake = G(x)
real_in = torch.cat([fake[:, 0:1], torch.ones(2, 1, 256, 256, device=device)], 1)
print(f"G output : {tuple(fake.shape)}   (B, 4=CBV,CBF,MTT,Tmax, H, W)")
print(f"D input  : {tuple(real_in.shape)}   (peta + channel indikator)")
print(f"D output : {tuple(D(real_in).shape)}   (peta probabilitas per-patch, RF 70x70)")
print(f"Param G  : {sum(p.numel() for p in G.parameters())/1e6:.2f}M")
print(f"Param D  : {sum(p.numel() for p in D.parameters())/1e6:.2f}M  (x4 discriminator)")

### 5. Custom Multi-Objective Loss (MAGIC)

Empat komponen loss diimplementasikan terpisah, lalu digabung dengan bobot $\lambda_1, \lambda_2, \lambda_3$ (persamaan (8) paper).

#### 5.a. GAN Loss ($\mathcal{L}_{GAN}$) — Modified Pix2Pix

$$\mathcal{L}_{GAN}=\frac{1}{N}\sum_{PM}\Big[\mathbb{E}[\log D_{PM}(PM)]+\mathbb{E}[\log(1-D_{PM}(G_{PM}(x)))]\Big]$$

- **D dimaksimalkan**: `BCE(D(real), 1)` + `BCE(D(fake.detach()), 0)` — dioptimalkan **lebih dulu lalu di-freeze** saat update G (detail implementasi paper).
- **G diminimalkan**: `BCE(D(fake), 1)` (non-saturating — setara objective GAN di atas).
- Input D = peta + channel indikator: ones untuk real, zeros untuk fake.
- Label smoothing real = 0.9 untuk stabilitas.

In [ ]:
class GANLoss(nn.Module):
    """Adversarial BCE — rata-rata atas N=4 peta & 4 discriminator."""
    def __init__(self, real_label=0.9):
        super().__init__()
        self.real_label = real_label
        self.bce = nn.BCEWithLogitsLoss()
    def _labels(self, t, val):
        return torch.full_like(t, val)
    def d_real_loss(self, d_real):
        """d_real: list logits D pada input REAL (peta + indikator ones)."""
        return torch.stack([self.bce(p, self._labels(p, self.real_label)) for p in d_real]).mean()
    def d_fake_losss(self, d_fake):
        """d_fake: list logits D pada input FAKE (peta + indikator zeros)."""
        return torch.stack([self.bce(p, self._labels(p, 0.0)) for p in d_fake]).mean()
    def g_adv_loss(self, d_fake):
        """G: minimalkan BCE(D(fake), 1) — non-saturating, setara pix2pix."""
        return torch.stack([self.bce(p, self._labels(p, self.real_label)) for p in d_fake]).mean()

#### 5.b. L1 Loss ($\mathcal{L}_{L1}$) — Structural Fidelity

$$\mathcal{L}_{L1}=\frac{1}{N}\sum_{PM}\mathbb{E}\big[\Vert PM-G_{PM}(x)\Vert_{1}\big]$$

In [ ]:
class StructureL1Loss(nn.Module):
    """L_L1: MAE antara peta sintetik & real, rata-rata atas 4 tipe peta."""
    def forward(self, gen, real):
        return F.l1_loss(gen, real)

#### 5.c. Extrema Loss ($\mathcal{L}_{EXT}$)

Menghukum berat error di **ROI yang menyimpang dari mean gambar** (jaringan sangat iskemik: CBV/CBF rendah, MTT/Tmax tinggi):

$$W_{x}=\Big(\frac{PM-\min(PM)}{\max(PM)-\min(PM)}-0.5\Big)^{2} \qquad H_{x}=\Big(\frac{G_{PM}(x)-\min(G_{PM}(x))}{\max(G_{PM}(x))-\min(G_{PM}(x))}-\frac{PM-\min(PM)}{\max(PM)-\min(PM)}\Big)^{2}$$

$$\mathcal{L}_{EXT}=\frac{1}{N}\sum_{PM}\mathbb{E}\big[W_{x}\odot H_{x}\big]$$

> **Coding note:** `min`/`max` dihitung **per gambar** (`dim=(2,3), keepdim=True`) + epsilon kecil untuk mencegah zero-division.

In [ ]:
def minmax_norm(t, eps=1e-6):
    """Normalisasi min-max PER GAMBAR. eps mencegah zero-division (peta konstan)."""
    tmin = t.amin(dim=(2, 3), keepdim=True)
    tmax = t.amax(dim=(2, 3), keepdim=True)
    return (t - tmin) / (tmax - tmin + eps)

class ExtremaLoss(nn.Module):
    """L_EXT = mean(W ⊙ H). W besar di region ekstrem, H = MSE ternormalisasi."""
    def forward(self, gen, real):
        W = (minmax_norm(real) - 0.5) ** 2
        H = (minmax_norm(gen) - minmax_norm(real)) ** 2
        return (W * H).mean()

#### 5.d. Multimodal Loss ($\mathcal{L}_{MML}$) — Central Volume Principle

$$\mathcal{L}_{MML}=\mathbb{E}\big[\Vert G_{CBF}(x)\times G_{MTT}(x)-CBV\Vert_{1}\big]$$

In [ ]:
class MultimodalLoss(nn.Module):
    """L_MML: ||G_CBF(x) ⊙ G_MTT(x) - CBV_real||_1 (Central Volume Principle)."""
    def __init__(self, idx_cbv=0, idx_cbf=1, idx_mtt=2):
        super().__init__()
        self.i_cbv, self.i_cbf, self.i_mtt = idx_cbv, idx_cbf, idx_mtt
    def forward(self, gen, real):
        return F.l1_loss(gen[:, self.i_cbf] * gen[:, self.i_mtt], real[:, self.i_cbv])

#### 5.e. Combined MAGIC Loss

Menerima tiga hyperparameter pembobot $\lambda_1, \lambda_2, \lambda_3$ (sesuai requirement): `MAGICLoss(lambda1, lambda2, lambda3)`.

In [ ]:
class MAGICLoss(nn.Module):
    """Total objective generator:
    G* = min_G max_D  L_GAN + λ1·L_L1 + λ2·L_EXT + λ3·L_MML
    """
    def __init__(self, lambda1=100.0, lambda2=0.1, lambda3=0.1, real_label=0.9):
        super().__init__()
        self.l1_w, self.l2_w, self.l3_w = lambda1, lambda2, lambda3
        self.gan = GANLoss(real_label)
        self.structure = StructureL1Loss()
        self.extrema = ExtremaLoss()
        self.multimodal = MultimodalLoss()
    def discriminator_loss(self, d_real, d_fake):
        """D step: max BCE(D(real),1) + BCE(D(fake.detach()),0)."""
        return self.gan.d_real_loss(d_real) + self.gan.d_fake_loss(d_fake)
    def generator_loss(self, gen, real, d_fake):
        """G step: adv + λ1·L1 + λ2·EXT + λ3·MML → dict komponen."""
        return {
            'g_adv': self.gan.g_adv_loss(d_fake),
            'l1':    self.l1_w * self.structure(gen, real),
            'ext':   self.l2_w * self.extrema(gen, real),
            'mml':   self.l3_w * self.multimodal(gen, real),
        }
    def generator_total(self, gen, real, d_fake):
        p = self.generator_loss(gen, real, d_fake)
        return p['g_adv'] + p['l1'] + p['ext'] + p['mml']

#### 5.f. Sanity Check Semua Loss

In [ ]:
gan = GANLoss()
dr = [torch.randn(2, 1, 30, 30)]; df = [torch.randn(2, 1, 30, 30)]
print(f"GAN  D(real)={gan.d_real_loss(dr):.4f}  D(fake)={gan.d_fake_loss(df):.4f}  G-adv={gan.g_adv_loss(df):.4f}")

mml = MultimodalLoss()
cbf = torch.rand(2, 256, 256); mtt = torch.rand(2, 256, 256); cbv = cbf * mtt
real = torch.stack([cbv, cbf, mtt, torch.rand_like(cbv)], dim=1)
gen_ok  = torch.stack([cbv, cbf, mtt, torch.rand_like(cbv)], dim=1)
gen_bad = torch.stack([cbv, cbf * 0.5, mtt, torch.rand_like(cbv)], dim=1)
print(f"MML  perfect={mml(gen_ok, real):.4f} (≈0)   violated={mml(gen_bad, real):.4f} (HIGH)")

ext = ExtremaLoss()
real_e = torch.zeros(1, 1, 256, 256)
for h in range(256):
    real_e[0, 0, h, :] = torch.linspace(0.5, 1.0, 256)
E_mid = real_e.clone(); E_mid[:, :, :, 120:136] += 0.1
E_ext = real_e.clone(); E_ext[:, :, :, 231:256] += 0.1
print(f"EXT  error@mid-intensity={ext(E_mid, real_e):.6f} (LOW)   error@extreme={ext(E_ext, real_e):.6f} (HIGH)")

### 6. Training & Evaluation Functions

**Loop GAN standar per iterasi** (D dioptimalkan lebih dulu, lalu G — sesuai paper):
1. `G(x)` → peta fake; `D(cat([real, ones]))` & `D(cat([fake.detach(), zeros]))` → **update D**
2. `G(x)` fresh → `D(cat([fake, ones]))` → `g_adv + λ1·L1 + λ2·EXT + λ3·MML` → **update G`

**Metrik** per tipe peta: **MAE, PSNR, SSIM, UQI** (Universal Quality Index — metrik utama paper, Wang & Bovik 2002).

In [ ]:
def uqi(gt, pred, eps=1e-8):
    """Universal Quality Index (Wang & Bovik 2002). 4·cov·mean_x·mean_y / ((σx²+σy²)(mx²+my²))."""
    gm, pm = gt.mean(), pred.mean()
    cov = ((gt - gm) * (pred - pm)).mean()
    denom = (gt.var() + pred.var()) * (gm ** 2 + pm ** 2) + eps
    return (4 * cov * gm * pm) / denom

def per_map_metrics(gen, real):
    """gen, real: (B,4,H,W) → {pm: (mae, psnr, ssim, uqi)} per tipe peta."""
    res = {}
    for i, pm in enumerate(PM_ORDER):
        g, r = gen[:, i], real[:, i]
        mae_v = F.l1_loss(g, r).item()
        mse = ((g - r) ** 2).mean()
        psnr_v = (10 * torch.log10(1.0 / (mse + 1e-8))).item()
        gn, rn = g.cpu().numpy(), r.cpu().numpy()
        ssim_v = np.mean([ssim_fn(rn[b], gn[b], data_range=1.0) for b in range(gn.shape[0])])
        uqi_v = np.mean([uqi(rn[b], gn[b]) for b in range(gn.shape[0])])
        res[pm] = (mae_v, psnr_v, ssim_v, uqi_v)
    return res

def val_psnr(G, loader):
    """Rata-rata PSNR atas semua peta & sampel (untuk model selection)."""
    G.eval(); p, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(device); real = batch['maps'].to(device)
            fake = G(x)
            for i in range(4):
                mse = ((fake[:, i] - real[:, i]) ** 2).mean()
                p += (10 * torch.log10(1.0 / (mse + 1e-8))).item() * real.shape[0]
            n += real.shape[0]
    return p / n

In [ ]:
def train_magic_v9(G, Ds, train_loader, val_loader, epochs=20, name='Model',
                   lr_g=2e-4, lr_d=2e-4, lambda1=100.0, lambda2=0.1, lambda3=0.1,
                   patience=8, md=0.001, ablation=None):
    """GAN training loop (D step → G step).
    ablation: None | 'no_ext' | 'no_mml' | 'gan_l1' (ablation studi paper)."""
    G = G.to(device); Ds = [d.to(device) for d in Ds]
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g, betas=(0.5, 0.999))
    opt_ds = [torch.optim.Adam(d.parameters(), lr=lr_d, betas=(0.5, 0.999)) for d in Ds]
    crit = MAGICLoss(lambda1, lambda2, lambda3).to(device)
    if ablation == 'no_ext':
        crit.l2_w = 0.0
    elif ablation == 'no_mml':
        crit.l3_w = 0.0
    elif ablation == 'gan_l1':
        crit.l2_w, crit.l3_w = 0.0, 0.0
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_g, mode='max', factor=0.5, patience=5)
    tag = name.lower().replace(' ', '_')
    hist = {'g_adv': [], 'l1': [], 'ext': [], 'mml': [], 'd_total': [], 'val_psnr': []}
    best_psnr, pc = 0.0, 0
    ab_txt = {None: 'FULL MAGIC (GAN+L1+EXT+MML)', 'no_ext': 'no-EXT (GAN+L1+MML)',
              'no_mml': 'no-MML (GAN+L1+EXT)', 'gan_l1': 'BASELINE (GAN+L1)'}
    print(f"\nTraining {name} [{ab_txt[ablation]}]...")
    print(f"{'Ep':>3}|{'g_adv':>7}|{'L1':>8}|{'EXT':>7}|{'MML':>7}|{'D':>7}|{'PSNR':>7}|{'Time':>6}")
    print('-' * 60)
    for ep in range(epochs):
        t0 = time.time(); G.train()
        for d in Ds:
            d.train()
        h = {'g_adv': 0.0, 'l1': 0.0, 'ext': 0.0, 'mml': 0.0, 'd_total': 0.0}
        for batch in train_loader:
            x = batch['x'].to(device); real = batch['maps'].to(device)
            # ---- 1) Discriminator step ----
            with torch.no_grad():
                fake = G(x)
            real_in = [torch.cat([real[:, i:i + 1], torch.ones_like(real[:, i:i + 1])], 1)
                       for i in range(len(Ds))]
            fake_in = [torch.cat([fake[:, i:i + 1], torch.zeros_like(fake[:, i:i + 1])], 1)
                       for i in range(len(Ds))]
            d_loss = crit.discriminator_loss([D(r) for D, r in zip(Ds, real_in)],
                                             [D(f) for D, f in zip(Ds, fake_in)])
            for od in opt_ds:
                od.zero_grad()
            d_loss.backward()
            for od in opt_ds:
                od.step()
            # ---- 2) Generator step ----
            fake = G(x)
            fake_in = [torch.cat([fake[:, i:i + 1], torch.ones_like(fake[:, i:i + 1])], 1)
                       for i in range(len(Ds))]
            d_fake = [D(f) for D, f in zip(Ds, fake_in)]
            p = crit.generator_loss(fake, real, d_fake)
            g_loss = p['g_adv'] + p['l1'] + p['ext'] + p['mml']
            opt_g.zero_grad()
            g_loss.backward()
            opt_g.step()
            h['g_adv'] += p['g_adv'].item()
            h['l1'] += p['l1'].item()
            h['ext'] += p['ext'].item()
            h['mml'] += p['mml'].item()
            h['d_total'] += d_loss.item()
        av = {k: v / len(train_loader) for k, v in h.items()}
        vp = val_psnr(G, val_loader)
        for k in ('g_adv', 'l1', 'ext', 'mml', 'd_total'):
            hist[k].append(av[k])
        hist['val_psnr'].append(vp)
        sched.step(vp)
        print(f"{ep + 1:>3}|{av['g_adv']:7.3f}|{av['l1']:8.3f}|{av['ext']:7.4f}|"
              f"{av['mml']:7.4f}|{av['d_total']:7.3f}|{vp:7.2f}|{time.time() - t0:6.1f}s")
        if vp > best_psnr + md:
            best_psnr, pc = vp, 0
            torch.save(G.state_dict(), f"{tag}_best.pth")
        else:
            pc += 1
        if pc >= patience:
            print(f'Early stop di epoch {ep + 1}')
            break
    return G, hist

In [ ]:
def eval_magic(models, test_loader):
    """Evaluasi per tipe peta: MAE / PSNR / SSIM / UQI."""
    print(f"{'Model':<24}|" + "".join([f"{pm:^28}" for pm in PM_ORDER]))
    print('-' * (24 + 28 * 4))
    res = {}
    for nm, G in models.items():
        G.eval(); agg = {pm: [0.0] * 4 for pm in PM_ORDER}; cnt = 0
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(device); real = batch['maps'].to(device)
                m = per_map_metrics(G(x), real)
                for pm in PM_ORDER:
                    for k in range(4):
                        agg[pm][k] += m[pm][k] * real.shape[0]
                cnt += real.shape[0]
        res[nm] = {pm: [v / cnt for v in agg[pm]] for pm in PM_ORDER}
        row = f"{nm:<24}|"
        for pm in PM_ORDER:
            row += f"{res[nm][pm][0]:.4f}/{res[nm][pm][1]:.1f}/{res[nm][pm][2]:.4f}/{res[nm][pm][3]:.4f}  "
        print(row)
    print("\nFormat per peta: MAE / PSNR / SSIM / UQI")
    return res

### 7. Inisialisasi

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders_v9('dataset')
gen_models = {}; histories = {}
N_MAPS = 4

### 8. Training & Ablation Study (sesuai paper)

Paper melaporkan ablasi: menghapus **multimodal loss**, **extrema loss**, atau keduanya menurunkan SSIM/UQI secara konsisten. Di sini kami reproduksi empat varian:

| Varian | Loss aktif | Lambda |
|---|---|---|
| `V9-Full` | GAN + L1 + EXT + MML | λ1=100, λ2=0.1, λ3=0.1 |
| `V9-noEXT` | GAN + L1 + MML | λ2 = 0 |
| `V9-noMML` | GAN + L1 + EXT | λ3 = 0 |
| `V9-Base` | GAN + L1 (≈ Pix2Pix) | λ2 = λ3 = 0 |

In [ ]:
G = MAGICGenerator().to(device)
Ds = [PatchGANDiscriminator().to(device) for _ in range(N_MAPS)]
G, h = train_magic_v9(G, Ds, train_loader, val_loader, epochs=5, name='MAGIC_V9-Full')
gen_models['MAGIC_V9-Full'] = G; histories['MAGIC_V9-Full'] = h

In [ ]:
G = MAGICGenerator().to(device)
Ds = [PatchGANDiscriminator().to(device) for _ in range(N_MAPS)]
G, h = train_magic_v9(G, Ds, train_loader, val_loader, epochs=5, name='MAGIC_V9-noEXT', ablation='no_ext')
gen_models['MAGIC_V9-noEXT'] = G; histories['MAGIC_V9-noEXT'] = h

In [ ]:
G = MAGICGenerator().to(device)
Ds = [PatchGANDiscriminator().to(device) for _ in range(N_MAPS)]
G, h = train_magic_v9(G, Ds, train_loader, val_loader, epochs=5, name='MAGIC_V9-noMML', ablation='no_mml')
gen_models['MAGIC_V9-noMML'] = G; histories['MAGIC_V9-noMML'] = h

In [ ]:
G = MAGICGenerator().to(device)
Ds = [PatchGANDiscriminator().to(device) for _ in range(N_MAPS)]
G, h = train_magic_v9(G, Ds, train_loader, val_loader, epochs=5, name='MAGIC_V9-Base', ablation='gan_l1')
gen_models['MAGIC_V9-Base'] = G; histories['MAGIC_V9-Base'] = h

### 9. Evaluasi Ablation

In [ ]:
print('=' * 80)
print('    ABLATION: Full vs no-EXT vs no-MML vs Base (GAN+L1)')
print('=' * 80)
print('\n--- Detail per peta ---')
res = eval_magic(gen_models, test_loader)

def avg_metric(nm, k):
    return float(np.mean([res[nm][pm][k] for pm in PM_ORDER]))

print('\n' + '=' * 80)
print('            PERBANDINGAN (rata-rata 4 peta, gain vs Full)')
print('=' * 80)
order = ['MAGIC_V9-Full', 'MAGIC_V9-noEXT', 'MAGIC_V9-noMML', 'MAGIC_V9-Base']
print(f"\n{'Model':<18}|{'PSNR':>8}|{'ΔPSNR':>8}|{'SSIM':>7}|{'UQI':>7}")
print('-' * 52)
f_psnr = avg_metric('MAGIC_V9-Full', 1)
for nm in order:
    p = avg_metric(nm, 1); s = avg_metric(nm, 2); u = avg_metric(nm, 3)
    d = p - f_psnr
    mark = ' (reference)' if nm == 'MAGIC_V9-Full' else ''
    print(f"{nm:<18}|{p:8.3f}|{d:+8.3f}|{s:7.4f}|{u:7.4f}{mark}")
print('\n' + '=' * 80)
print('  >>> Jika Full > noEXT/noMML/Base: EXT + MML berkontribusi (konsisten dgn paper). <<<')
print('=' * 80)

### 10. Visualisasi

In [ ]:
def vis_generation(models, loader, n=1):
    names = list(models.keys())
    cols = 3 + 4 + len(names) * 4     # 3 slice NCCT + GT(4) + per-model 4 peta
    rows = n * len(names)
    plt.figure(figsize=(cols * 2.4, rows * 2.4))
    with torch.no_grad():
        for i in range(n):
            batch = next(iter(loader))
            x = batch['x'].to(device); real = batch['maps'].to(device)
            fake_all = {nm: models[nm](x) for nm in names}
            for r, nm in enumerate(names):
                row = i * len(names) + r
                if r == 0:
                    for k, lbl in zip(range(3), ['NCCT p', 'NCCT m', 'NCCT n']):
                        plt.subplot(rows, cols, row * cols + k + 1)
                        plt.imshow(x[0, k].cpu().numpy(), cmap='gray')
                        plt.title(lbl, fontsize=8); plt.axis('off')
                    for j, pm in enumerate(PM_ORDER):
                        plt.subplot(rows, cols, row * cols + 4 + j)
                        plt.imshow(real[0, j].cpu().numpy(), cmap='viridis')
                        plt.title(f'GT {pm}', fontsize=8); plt.axis('off')
                c = 3 + 4 + r * 4
                for j, pm in enumerate(PM_ORDER):
                    plt.subplot(rows, cols, row * cols + c + j)
                    plt.imshow(fake_all[nm][0, j].cpu().numpy(), cmap='viridis')
                    plt.title(f'{nm.split("V9-")[1]}\\n{pm}', fontsize=7); plt.axis('off')
    plt.tight_layout(); plt.show()

if gen_models:
    vis_generation(gen_models, test_loader)

#### 10.b. PILO — Interaktif Rasio Struktur vs Perfusi (α)

Modul PILO (Physician-in-the-Loop) paper memungkinkan klinisi mengatur **rasio informasi struktural NCCT vs perfusi sintetik** saat inferensi. Demo blending `α·map + (1-α)·NCCT_mid` untuk α = 0, 0.5, 1.

In [ ]:
def pilo_demo(G, loader, alphas=(0.0, 0.5, 1.0)):
    with torch.no_grad():
        batch = next(iter(loader))
        x = batch['x'].to(device); real = batch['maps'].to(device)
        gen = G(x)
        mid = x[:, 1:2].cpu().numpy()
        fig, axes = plt.subplots(len(PM_ORDER), len(alphas) + 1, figsize=(4 * (len(alphas) + 1), 4 * len(PM_ORDER)))
        for j, pm in enumerate(PM_ORDER):
            axes[j, 0].imshow(real[0, j].cpu().numpy(), cmap='viridis')
            axes[j, 0].set_title(f'GT {pm}', fontsize=9); axes[j, 0].axis('off')
            for a, alpha in enumerate(alphas):
                blend = alpha * gen[0, j].cpu().numpy() + (1 - alpha) * mid[0, 0]
                axes[j, a + 1].imshow(blend, cmap='viridis')
                axes[j, a + 1].set_title(f'α={alpha:.1f}', fontsize=9); axes[j, a + 1].axis('off')
        plt.suptitle('PILO: rasio struktur-NCCT vs perfusi-sintetik', fontsize=12)
        plt.tight_layout(); plt.show()

if 'MAGIC_V9-Full' in gen_models:
    pilo_demo(gen_models['MAGIC_V9-Full'], test_loader)

### 11. Training Curves

In [ ]:
def plot_curves9(hist):
    fig, ax = plt.subplots(2, 3, figsize=(16, 9))
    metrics = ['g_adv', 'l1', 'ext', 'mml', 'd_total', 'val_psnr']
    titles = ['G Adversarial', 'L1 (×λ1)', 'Extrema (×λ2)', 'Multimodal (×λ3)', 'D Total', 'Val PSNR']
    for a, met, t in zip(ax.ravel(), metrics, titles):
        for k, h in hist.items():
            style = '-' if 'Full' in k else '--'
            a.plot(h[met], style, label=k, lw=2)
        a.set(xlabel='Epoch', ylabel=t, title=t)
        a.legend(fontsize=7); a.grid(alpha=0.3)
    plt.suptitle('MAGIC Ablation: Full / no-EXT / no-MML / Base', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

if histories:
    plot_curves9(histories)

### 12. Hyperparameter Tuning — λ1, λ2, λ3

Sweep singkat (5 epoch, MAGICGenerator) untuk mencari kombinasi bobot terbaik; dipilih yang menghasilkan PSNR validasi tertinggi.

In [ ]:
configs = [
    {'lambda1': 100.0, 'lambda2': 0.1, 'lambda3': 0.1},
    {'lambda1': 100.0, 'lambda2': 0.5, 'lambda3': 0.1},
    {'lambda1': 100.0, 'lambda2': 0.1, 'lambda3': 0.5},
    {'lambda1': 10.0,  'lambda2': 0.1, 'lambda3': 0.1},
    {'lambda1': 100.0, 'lambda2': 1.0, 'lambda3': 1.0},
    {'lambda1': 50.0,  'lambda2': 0.5, 'lambda3': 0.5},
]
tune = {}
print('=' * 66)
print('   Lambda Tuning (5 epoch each, MAGICGenerator)')
print('=' * 66)
for i, cfg in enumerate(configs):
    G = MAGICGenerator().to(device)
    Ds = [PatchGANDiscriminator().to(device) for _ in range(N_MAPS)]
    print(f"\nConfig {i + 1}: λ1={cfg['lambda1']}  λ2={cfg['lambda2']}  λ3={cfg['lambda3']}")
    G, h = train_magic_v9(G, Ds, train_loader, val_loader, epochs=5, name=f'tune{i}', **cfg)
    tune[i] = {'psnr': h['val_psnr'][-1], 'cfg': cfg}
print('\n' + '=' * 66)
print(f"{'Cfg':<6}|{'λ1':<8}|{'λ2':<6}|{'λ3':<6}|{'PSNR':<8}")
print('-' * 36)
bi = max(tune, key=lambda k: tune[k]['psnr'])
for i, d in tune.items():
    c = d['cfg']
    mark = '  <<<' if i == bi else ''
    print(f"{i:<6}|{c['lambda1']:<8}|{c['lambda2']:<6}|{c['lambda3']:<6}|{d['psnr']:<8.3f}{mark}")
bc = tune[bi]['cfg']
print(f"\n>>> Best: λ1={bc['lambda1']}  λ2={bc['lambda2']}  λ3={bc['lambda3']}")
print('=' * 66)

### 13. Ringkasan & Kesimpulan

**V9 (MAGIC GAN) — kesesuaian dengan paper arXiv:2505.22673v2:**

| Aspek | Implementasi notebook |
|-------|-----------------------|
| Dataset | **ISLES'24 publik** (Hugging Face / Zenodo) — NCCT + peta CBV/CBF/MTT/Tmax coregistrasi `space-ncct` (Tmax menggantikan TTP) |
| Input | **Pseudo-RGB NCCT** — 3 slice z bertetangga (±4 mm) di-stack (3×256×256) |
| Normalisasi | NCCT window 40/80 HU; peta dengan rentang fisiologis (paper §2.1.2) |
| Generator | **Shared encoder + 4 decoder head** (per tipe peta) + modul **PILO** |
| Discriminator | **PatchGAN 70×70 per peta**, input peta + channel indikator ones/zeros |
| GAN loss | Modified Pix2Pix, BCE rata-rata atas 4 peta; D dioptimalkan sebelum G |
| L1 loss | MAE rata-rata atas 4 peta |
| Extrema loss | $W=(norm(PM)-0.5)^2$, $H=(norm(G)-norm(PM))^2$, per-gambar + ε |
| Multimodal loss | $||G_{CBF}\odot G_{MTT}-CBV||_1$ (Central Volume Principle) |
| Metrik | MAE, PSNR, SSIM, **UQI** (metrik utama paper) |

**Referensi:**
- Khan, W., et al. "Diagnostically Competitive Performance of a Physiology-Informed Generative Multi-Task Network for Contrast-Free CT Perfusion." *arXiv:2505.22673v2* (2026)
- Riedel, O. E., et al. "ISLES'24 – A Real-World Longitudinal Multimodal Stroke Dataset." *arXiv:2408.11142* (2024), Zenodo 10.5281/zenodo.17652035, CC BY-NC-SA 4.0
- Isola et al. "Image-to-Image Translation with Conditional Adversarial Networks (Pix2Pix)" — CVPR 2017 (PatchGAN 70×70)
- Ronneberger et al. "U-Net" — MICCAI 2015 (shared encoder / skip connections)
- Hoeffner et al. "Cerebral Perfusion CT" — Radiology 2004 (Central Volume Principle)
- Wang & Bovik, "A Universal Image Quality Index" — IEEE SPL 2002